# OpenADMET CYP, GPU experiments

Every TabPFN run in this project so far was made on CPU with `fit_mode="low_memory"`,
`memory_saving_mode=True`, `n_jobs=2` and PCA to 200 components. TabPFN is the strongest
arm at all four endpoints and has never been run properly. Experiment A isolates each of
those compromises one at a time.

Folds, holdout mask and features are the ones the project already uses, so every number
here drops straight into the tables in `STATUS.md`.

**Runtime, GPU, T4.** Check it with the cell below before anything else.

The target: development out-of-fold macro Spearman is 0.6306 and needs about 0.673 for a
top-40 finish on the blind board. Per isoform the shipped blend is CYP1A2 0.5791,
CYP2C9 0.7076, CYP2D6 0.4263, CYP3A4 0.8094.

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "NO GPU")

## Setup

Nothing to upload. The data is embedded in the cell below and unpacks to
`/content/cache`. Results collect in `/content/results` and the last cell
downloads them as a single zip.

In [ ]:
# The data lives in Drive at cyp_payloads/: labels including the TDI columns, SMILES, scaffold fold ids,
# the locked holdout mask, the frozen Chemprop embeddings (float16, worst
# principal component still correlating 0.99988 with the float32 original),
# the qHTS probability columns, and the cached tree and TabPFN out-of-fold
# predictions so a blend can be checked here rather than only locally.
#
# The qHTS columns are embedded rather than recomputed because building them
# took a PubChem crawl over roughly 16,000 compounds. The 2,264 ECFP and
# descriptor columns are left out instead of embedded, since RDKit rebuilds
# them from SMILES in a couple of minutes if anything here ends up needing
# them.
PAYLOAD_FILE = '/content/drive/MyDrive/cyp_payloads/mini3_d3108f9c6e.tgz'

import os, tarfile
os.makedirs('/content/cache', exist_ok=True)
os.makedirs('/content/results', exist_ok=True)
OUTDIR = '/content/results'
from google.colab import drive
drive.mount('/content/drive')
with tarfile.open(PAYLOAD_FILE) as t:
    t.extractall('/content/cache')
print(sorted(os.listdir('/content/cache')))

In [ ]:
!pip -q install tabpfn==2.2.1

In [ ]:
import json, time, numpy as np, pandas as pd, torch
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

CACHE = Path('/content/cache')
OUT = Path(OUTDIR)
ISOFORMS = ['CYP1A2', 'CYP2C9', 'CYP2D6', 'CYP3A4']
N_FOLDS = 5
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

meta = pd.read_csv(CACHE / 'meta_train.csv.gz')
hold = np.load(CACHE / 'holdout_mask.npy')
G = np.load(CACHE / 'scaffold_groups.npy', allow_pickle=True)
EMB = np.load(CACHE / 'emb_medium_inhib.npy').astype(np.float32)
QH = np.load(CACHE / 'qhts_tr.npy')
AUX = np.load(CACHE / 'aux_train.npy')
SMILES_TR = meta['SMILES'].to_numpy()

# What the project currently ships, for reference in every table below.
SHIPPED = {'CYP1A2': 0.5791, 'CYP2C9': 0.7076, 'CYP2D6': 0.4263, 'CYP3A4': 0.8094}

def dev_index(iso):
    y = meta[f'{iso}_pIC50_direct_inhibition'].to_numpy(float)
    return y, np.flatnonzero(np.isfinite(y) & ~hold)

def pca_fit(a, b, n):
    n = min(n, a.shape[0], a.shape[1])
    p = Pipeline([('i', SimpleImputer(strategy='median')), ('s', StandardScaler()),
                  ('p', PCA(n_components=n, random_state=42))])
    return p.fit_transform(a).astype(np.float32), p.transform(b).astype(np.float32)

RESULTS = []
def record(**kw):
    RESULTS.append(kw)
    with open(OUT / 'results.json', 'w') as f:
        json.dump(RESULTS, f, indent=2)

print('device', DEVICE, '| dev sizes',
      {i: len(dev_index(i)[1]) for i in ISOFORMS})

## Experiment A, TabPFN without the CPU compromises

Four arms, each removing one more of the restrictions the CPU forced:

| arm | device | PCA | fit_mode | memory saving | estimators |
| :--- | :--- | ---: | :--- | :--- | ---: |
| `cpu_equiv` | cuda | 200 | low_memory | on | default |
| `full_pca` | cuda | 200 | fit_preprocessors | off | default |
| `full_raw` | cuda | none (605 cols) | fit_preprocessors | off | default |
| `full_raw_ens` | cuda | none (609 cols, with primary screen) | fit_preprocessors | off | 16 |

`cpu_equiv` is the control. It should land near the numbers already in `STATUS.md`
(CYP1A2 0.5610, CYP2C9 0.7189, CYP2D6 0.4219, CYP3A4 0.8074). If it does not, the
comparison is not measuring what it claims to and the rest of the sheet is void.

In [ ]:
from tabpfn import TabPFNRegressor

ARMS = {
    'cpu_equiv':     dict(pca=200, feats='emb_q',     low=True,  n_est=None),
    'full_pca':      dict(pca=200, feats='emb_q',     low=False, n_est=None),
    'full_raw':      dict(pca=None, feats='emb_q',     low=False, n_est=None),
    'full_raw_ens':  dict(pca=None, feats='emb_q_aux', low=False, n_est=16),
}

def features(kind):
    return {'emb_q': np.hstack([EMB, QH]),
            'emb_q_aux': np.hstack([EMB, QH, AUX])}[kind]

def make(cfg):
    kw = dict(device=DEVICE, random_state=42, ignore_pretraining_limits=True)
    if cfg['low']:
        kw.update(fit_mode='low_memory', memory_saving_mode=True, n_jobs=2)
    if cfg['n_est']:
        kw['n_estimators'] = cfg['n_est']
    return TabPFNRegressor(**kw)

def run_arm(name, cfg, isoforms=ISOFORMS):
    S = features(cfg['feats'])
    out = {}
    for iso in isoforms:
        y, dev = dev_index(iso)
        t0 = time.time()
        oof = np.full(len(dev), np.nan)
        for tr, va in GroupKFold(N_FOLDS).split(dev, dev, G[dev]):
            a, b = dev[tr], dev[va]
            if cfg['pca']:
                Xa, Xb = pca_fit(S[a], S[b], cfg['pca'])
            else:
                Xa, Xb = S[a].astype(np.float32), S[b].astype(np.float32)
            m = make(cfg); m.fit(Xa, y[a])
            oof[va] = m.predict(Xb)
        rho = float(spearmanr(y[dev], oof).statistic)
        out[iso] = rho
        np.save(OUT / f'tabpfn_{name}_{iso}.npy', oof)
        record(experiment='A', arm=name, iso=iso, spearman=rho,
               shipped=SHIPPED[iso], seconds=round(time.time() - t0, 1))
        print(f'  {name:14s} {iso} {rho:.4f}  (shipped {SHIPPED[iso]:.4f}, '
              f'{rho - SHIPPED[iso]:+.4f})  {time.time()-t0:.0f}s', flush=True)
    print(f'  {name:14s} MACRO {np.mean(list(out.values())):.4f} '
          f'(shipped {np.mean(list(SHIPPED.values())):.4f})\n', flush=True)
    return out

A = {}
for name, cfg in ARMS.items():
    print(name, cfg, flush=True)
    A[name] = run_arm(name, cfg)

### Read the control first

If `cpu_equiv` does not reproduce the CPU numbers to within about 0.01, stop and say so
rather than reporting the other three arms. Everything downstream assumes the folds and
features match what the project already measured.

In [ ]:
import pandas as pd
tab = pd.DataFrame(A).T
tab['MACRO'] = tab.mean(axis=1)
tab.loc['shipped'] = list(SHIPPED.values()) + [np.mean(list(SHIPPED.values()))]
tab.loc['cpu_reference'] = [0.5610, 0.7189, 0.4219, 0.8074,
                            np.mean([0.5610, 0.7189, 0.4219, 0.8074])]
display(tab.round(4))
print('\ncontrol drift vs the CPU reference:')
print((tab.loc['cpu_equiv'] - tab.loc['cpu_reference']).round(4).to_string())

## Experiment B, D-MPNN at ensemble scale

The prior D-MPNN work used three seeds and no hyperparameter search, because each run
cost hours on CPU. Ensemble size is the most reliable source of ranking gain in this
literature and the project has never bought any of it.

This trains a compact D-MPNN directly in PyTorch rather than through Chemprop, so there
is no version friction with Colab. Message passing over bond features, mean pooling, a
two-layer head, scaffold folds identical to everything else.

In [ ]:
!pip -q install rdkit

In [ ]:
import torch, torch.nn as nn
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')

SMILES_TR = meta['SMILES'].to_numpy()

ATOMS = list(range(1, 54)) + [0]
HYB = list(range(8))

def atom_feat(a):
    v = [0.0] * (len(ATOMS) + len(HYB) + 6)
    n = a.GetAtomicNum()
    v[ATOMS.index(n) if n in ATOMS else len(ATOMS) - 1] = 1.0
    h = int(a.GetHybridization())
    v[len(ATOMS) + (h if h < len(HYB) else 0)] = 1.0
    o = len(ATOMS) + len(HYB)
    v[o] = a.GetDegree() / 4.0
    v[o + 1] = a.GetFormalCharge()
    v[o + 2] = a.GetTotalNumHs() / 4.0
    v[o + 3] = float(a.GetIsAromatic())
    v[o + 4] = float(a.IsInRing())
    v[o + 5] = a.GetMass() / 100.0
    return v

BT = [Chem.BondType.SINGLE, Chem.BondType.DOUBLE, Chem.BondType.TRIPLE,
      Chem.BondType.AROMATIC]

def bond_feat(b):
    v = [0.0] * (len(BT) + 2)
    t = b.GetBondType()
    if t in BT:
        v[BT.index(t)] = 1.0
    v[len(BT)] = float(b.GetIsConjugated())
    v[len(BT) + 1] = float(b.IsInRing())
    return v

def to_graph(smi):
    m = Chem.MolFromSmiles(smi)
    if m is None or m.GetNumAtoms() == 0:
        m = Chem.MolFromSmiles('C')
    x = np.array([atom_feat(a) for a in m.GetAtoms()], np.float32)
    src, dst, ef = [], [], []
    for b in m.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        f = bond_feat(b)
        src += [i, j]; dst += [j, i]; ef += [f, f]
    if not src:
        src, dst, ef = [0], [0], [[0.0] * (len(BT) + 2)]
    return x, np.array(src), np.array(dst), np.array(ef, np.float32)

print('featurising', len(SMILES_TR), 'molecules', flush=True)
GRAPHS = [to_graph(s) for s in SMILES_TR]
ADIM, BDIM = GRAPHS[0][0].shape[1], GRAPHS[0][3].shape[1]
print('atom dim', ADIM, 'bond dim', BDIM)

In [ ]:
def batch(idx):
    xs, ss, ds, es, bid, rev = [], [], [], [], [], []
    off = eoff = 0
    for k, i in enumerate(idx):
        x, s, d, e = GRAPHS[i]
        xs.append(x); ss.append(s + off); ds.append(d + off); es.append(e)
        bid.append(np.full(len(x), k))
        # edges were added in (i->j, j->i) pairs, so the reverse of edge e is e ^ 1
        r = np.arange(len(s)) ^ 1
        rev.append(np.minimum(r, len(s) - 1) + eoff)
        off += len(x); eoff += len(s)
    T = lambda a, t=torch.float32: torch.as_tensor(np.concatenate(a), dtype=t, device=DEVICE)
    return (T(xs), T(ss, torch.long), T(ds, torch.long), T(es),
            T(bid, torch.long), len(idx), T(rev, torch.long))

class DMPNN(nn.Module):
    def __init__(self, hid=300, depth=3, drop=0.0):
        super().__init__()
        self.depth = depth
        self.wi = nn.Linear(ADIM + BDIM, hid)
        self.wh = nn.Linear(hid, hid)
        self.wo = nn.Linear(ADIM + hid, hid)
        self.head = nn.Sequential(nn.Dropout(drop), nn.Linear(hid, hid), nn.ReLU(),
                                  nn.Dropout(drop), nn.Linear(hid, 1))
        self.act = nn.ReLU()

    def forward(self, x, src, dst, ef, bid, n, rev):
        h0 = self.act(self.wi(torch.cat([x[src], ef], 1)))
        h = h0
        for _ in range(self.depth - 1):
            # messages into the source atom, minus the one coming back along this
            # same bond, which is what makes it directed rather than a plain GNN
            agg = torch.zeros_like(h).index_add_(0, dst, h)
            h = self.act(h0 + self.wh(agg[src] - h[rev]))
        msg = torch.zeros(x.size(0), h.size(1), device=DEVICE).index_add_(0, dst, h)
        a = self.act(self.wo(torch.cat([x, msg], 1)))
        g = torch.zeros(n, a.size(1), device=DEVICE).index_add_(0, bid, a)
        cnt = torch.zeros(n, 1, device=DEVICE).index_add_(
            0, bid, torch.ones(x.size(0), 1, device=DEVICE))
        return self.head(g / cnt.clamp(min=1)).squeeze(1)

def train_one(tr, va, y, hid, depth, drop, lr, epochs, seed, bs=64):
    torch.manual_seed(seed); np.random.seed(seed)
    m = DMPNN(hid, depth, drop).to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    sch = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=lr, total_steps=epochs * max(1, len(tr) // bs + 1))
    mu, sd = y[tr].mean(), y[tr].std()
    for _ in range(epochs):
        m.train()
        perm = np.random.permutation(len(tr))
        for k in range(0, len(perm), bs):
            b = tr[perm[k:k + bs]]
            if len(b) < 2:
                continue
            opt.zero_grad()
            p = m(*batch(b))
            t = torch.as_tensor((y[b] - mu) / sd, dtype=torch.float32, device=DEVICE)
            loss = nn.functional.mse_loss(p, t)
            loss.backward(); opt.step(); sch.step()
    m.eval()
    with torch.no_grad():
        out = []
        for k in range(0, len(va), 256):
            out.append(m(*batch(va[k:k + 256])).cpu().numpy())
    return np.concatenate(out) * sd + mu

In [ ]:
GRID = [dict(hid=300, depth=3, drop=0.0,  lr=1e-3, epochs=50),
        dict(hid=600, depth=4, drop=0.1,  lr=8e-4, epochs=60),
        dict(hid=300, depth=5, drop=0.15, lr=1e-3, epochs=60),
        dict(hid=900, depth=3, drop=0.2,  lr=5e-4, epochs=80)]
SEEDS = [0, 1, 2, 3, 4]

B = {}
for iso in ISOFORMS:
    y, dev = dev_index(iso)
    print(f'\n{iso} n={len(dev)}', flush=True)
    best = None
    for gi, g in enumerate(GRID):
        t0 = time.time()
        oof = np.zeros(len(dev))
        for tr, va in GroupKFold(N_FOLDS).split(dev, dev, G[dev]):
            acc = np.zeros(len(va))
            for s in SEEDS:
                acc += train_one(dev[tr], dev[va], y, seed=s, **g)
            oof[va] = acc / len(SEEDS)
        rho = float(spearmanr(y[dev], oof).statistic)
        record(experiment='B', iso=iso, grid=gi, **g, n_seeds=len(SEEDS),
               spearman=rho, shipped=SHIPPED[iso], seconds=round(time.time() - t0, 1))
        print(f'  grid{gi} {g} -> {rho:.4f}  ({time.time()-t0:.0f}s)', flush=True)
        if best is None or rho > best[0]:
            best = (rho, gi, oof)
    np.save(OUT / f'dmpnn_{iso}.npy', best[2])
    B[iso] = best[0]
    print(f'  best grid{best[1]} {best[0]:.4f} vs shipped blend {SHIPPED[iso]:.4f}')
print('\nMACRO D-MPNN', round(float(np.mean(list(B.values()))), 4))

## Experiment C, MoLFormer-XL fine-tuned end to end

The CPU test used frozen mean-pooled hidden states and came to macro +0.0011. That is
the weakest possible use of a transformer. This fine-tunes the whole encoder on each
endpoint with the same scaffold folds.

MoLFormer loads with `trust_remote_code=True`, so IBM's own modelling code runs in this
runtime. It is their official repository, and it is downloaded code.

In [ ]:
!pip -q install transformers

In [ ]:
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel

REPO = 'ibm-research/MoLFormer-XL-both-10pct'
tok = AutoTokenizer.from_pretrained(REPO, trust_remote_code=True)


class Regressor(nn.Module):
    """Own head on the encoder, since the repo registers no regression head."""

    def __init__(self):
        super().__init__()
        self.enc = AutoModel.from_pretrained(REPO, trust_remote_code=True)
        d = self.enc.config.hidden_size
        self.head = nn.Sequential(nn.Dropout(0.1), nn.Linear(d, 1))

    def forward(self, **b):
        h = self.enc(**b).last_hidden_state
        m = b['attention_mask'].unsqueeze(-1).float()
        return self.head((h * m).sum(1) / m.sum(1).clamp(min=1)).squeeze(-1)

def finetune(tr, va, y, epochs=12, lr=3e-5, bs=32, seed=0):
    torch.manual_seed(seed)
    m = Regressor().to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=0.01)
    mu, sd = y[tr].mean(), y[tr].std()
    enc = lambda idx: tok(list(SMILES_TR[idx]), padding=True, truncation=True,
                          max_length=202, return_tensors='pt').to(DEVICE)
    for _ in range(epochs):
        m.train()
        perm = np.random.permutation(len(tr))
        for k in range(0, len(perm), bs):
            b = tr[perm[k:k + bs]]
            opt.zero_grad()
            p = m(**enc(b))
            t = torch.as_tensor((y[b] - mu) / sd, dtype=torch.float32, device=DEVICE)
            nn.functional.mse_loss(p, t).backward()
            opt.step()
    m.eval()
    out = []
    with torch.no_grad():
        for k in range(0, len(va), 64):
            out.append(m(**enc(va[k:k + 64])).cpu().numpy())
    del m; torch.cuda.empty_cache()
    return np.concatenate(out) * sd + mu

C = {}
for iso in ISOFORMS:
    y, dev = dev_index(iso)
    t0 = time.time()
    oof = np.zeros(len(dev))
    for tr, va in GroupKFold(N_FOLDS).split(dev, dev, G[dev]):
        oof[va] = finetune(dev[tr], dev[va], y)
    rho = float(spearmanr(y[dev], oof).statistic)
    np.save(OUT / f'molformer_ft_{iso}.npy', oof)
    C[iso] = rho
    record(experiment='C', iso=iso, spearman=rho, shipped=SHIPPED[iso],
           seconds=round(time.time() - t0, 1))
    print(f'{iso} fine-tuned {rho:.4f} vs frozen 0.40/0.50/0.28/0.63, '
          f'shipped blend {SHIPPED[iso]:.4f}  ({time.time()-t0:.0f}s)', flush=True)
print('\nMACRO MoLFormer fine-tuned', round(float(np.mean(list(C.values()))), 4))

## What comes back

The last cell downloads `cyp_gpu_results.zip`: `results.json` plus the per-isoform
out-of-fold prediction arrays. The arrays matter as much as the scores, because they
let the blend be refitted locally against the trees and the existing TabPFN arm, which
is the only way to tell whether a stronger single model actually moves the blend. Six
separate attempts have now gained at the single-model level and vanished at the blend
level, so a good score in this notebook is not yet a result.

In [ ]:
import pandas as pd, json, shutil
r = pd.DataFrame(json.load(open(OUT / 'results.json')))
display(r)

# One zip to hand back: the scores and every out-of-fold array.
shutil.make_archive('/content/cyp_gpu_results', 'zip', OUT)
from google.colab import files
files.download('/content/cyp_gpu_results.zip')